### ============================================================
### DATA COLLECTION NOTEBOOK — DO NOT RE-RUN
### Outputs are already saved to /data/. Re-running requires API connection and will overwrite archived raw data.
### ============================================================

# 5
In this notebook I use the same year range to collect random, control group debates and also normalize speaker names to merge with the members of parliament.

In [2]:
import pandas as pd
import re
from unidecode import unidecode
import requests
import xml.etree.ElementTree as ET
import random
from time import sleep

In [1]:
# Load members data
df_members = pd.read_csv('data/members_with_gender_guess.csv')

# Normalization Function from notebook
def normalize_speaker(speaker):
    if not isinstance(speaker, str) or not speaker.strip():
        return None
    paren_match = re.search(r'\((.*?)\)', speaker)
    name = paren_match.group(1) if paren_match else speaker
    name = re.sub(
        r'\b(Deputy|Senator|Minister(?: of State)?|The Taoiseach|The T[áa]naiste|'
        r'An Ceann Comhairle|An Leas-Cheann Comhairle|An Cathaoirleach|Mr\.?|Ms\.?|Mrs\.?|Dr\.?|'
        r'Professor|Chair|Chairman|Rev|Fr)\b',
        '', name, flags=re.IGNORECASE
    )
    name = re.sub(r'Minister [^()]*', '', name, flags=re.IGNORECASE)
    name = name.replace("’", "'").replace("`", "'").replace("‘", "'")
    name = re.sub(r"\bo\s+([a-z])", r"o'\1", name, flags=re.IGNORECASE)
    name = re.sub(r"[.,]", "", name)
    name = re.sub(r"\s+", " ", name).strip()
    name = unidecode(name).lower()
    return name if name else None

# Role Lookup
# from wikipedia
role_data = [
    # Role, Start Date, End Date, Normalized Full Name
    ('The Taoiseach', '2008-05-07', '2011-03-09', 'brian cowen'),
    ('The Taoiseach', '2011-03-09', '2017-06-14', 'enda kenny'),
    ('The Taoiseach', '2017-06-14', '2020-06-27', 'leo varadkar'),
    ('The Taoiseach', '2020-06-27', '2022-12-17', 'micheal martin'),
    ('The Taoiseach', '2022-12-17', '2024-04-09', 'leo varadkar'),
    ('The Taoiseach', '2024-04-09', '2025-01-23', 'simon harris'),
    ('The Taoiseach', '2025-01-23', None, 'micheal martin'), # Example of a current role
    ('An Leas-Cheann Comhairle', '2007-06-26', '2011-03-09', 'brendan howlin'),
    ('An Leas-Cheann Comhairle', '2011-03-31', '2016-03-10', 'michael p. kitt'),
    ('An Leas-Cheann Comhairle', '2016-07-07', '2020-01-14', 'pat the cope gallagher'),
    ('An Leas-Cheann Comhairle', '2020-07-23', '2024-11-08', 'catherine connolly'),
    ('An Leas-Cheann Comhairle', '2025-02-19', None, 'john mcguiness'),
    ('An Cathaoirleach', '2007-09-13', '2011-05-25', 'pat moylan'),
    ('An Cathaoirleach', '2011-05-25', '2016-06-08', 'paddy burke'),
    ('An Cathaoirleach', '2016-06-08', '2020-06-29', "denis o'donovan"),
    ('An Cathaoirleach', '2020-06-29', '2022-12-16', "mark daly"),
    ('An Cathaoirleach', '2022-12-16', '2024-11-30', "jerry buttimer"),
    ('An Cathaoirleach', '2025-02-12', None, "mark daly"),
    ('The Tánaiste', '2008-05-07', '2011-03-09', 'mary coughlan'),
    ('The Tánaiste', '2011-03-09', '2014-07-04', 'eamon gilmore'),
    ('The Tánaiste', '2014-07-04', '2016-05-06', 'joan burton'),
    ('The Tánaiste', '2016-05-06', '2017-11-28', 'frances fitzgerald'),
    ('The Tánaiste', '2017-11-30', '2020-06-27', 'simon coveney'),
    ('The Tánaiste', '2020-06-27', '2022-12-17', 'leo varadkar'),
    ('The Tánaiste', '2022-12-17', '2025-01-23', 'micheal martin'),
    ('The Tánaiste', '2025-01-23', None, 'simon harris'),
    ('An Ceann Comhairle', '2007-06-14', '2009-10-13', "john o'donoghue"),
    ('An Ceann Comhairle', '2009-10-13', '2011-03-09', 'seamus kirk'),
    ('An Ceann Comhairle', '2011-03-09', '2016-03-10', 'sean barrett'),
    ('An Ceann Comhairle', '2016-03-10', '2024-12-18', 'sean o fearghail'),
    ('An Ceann Comhairle', '2024-12-18', None, 'verona murphy'),
    ('Chairman', '2007-06-14', '2009-10-13', "john o'donoghue"),
    ('Chairman', '2009-10-13', '2011-03-09', 'seamus kirk'),
    ('Chairman', '2011-03-09', '2016-03-10', 'sean barrett'),
    ('Chairman', '2016-03-10', '2024-12-18', 'sean o fearghail'),
    ('Chairman', '2024-12-18', None, 'verona murphy')
]
role_lookup = pd.DataFrame(role_data, columns=['role', 'start_date', 'end_date', 'fullName_norm'])
role_lookup['start_date'] = pd.to_datetime(role_lookup['start_date'])
role_lookup['end_date'] = pd.to_datetime(role_lookup['end_date']).fillna(pd.Timestamp('2099-12-31'))

# Normalize members for merging
df_members['fullName_norm'] = df_members['fullName'].apply(lambda x: unidecode(x).lower().strip() if isinstance(x, str) else None)

print("Setup Complete.")

Setup Complete.


In [2]:
# Parameters
YEARS = list(range(2010, 2026)) # Adjust range as needed
ns = {'akn': 'http://docs.oasis-open.org/legaldocml/ns/akn/3.0/CSD13'}
speech_pool = []

for year in YEARS:
    metadata_file = f"raw_data/final_debates_{year}_metadata.csv"

    try:
        df_meta = pd.read_csv(metadata_file).sample(frac=0.2) # Sample 20% of debates to build a large enough pool
    except: continue

    print(f"[{year}] Fetching speeches...")
    for _, row in df_meta.iterrows():
        try:
            r = requests.get(row['xml_uri'])
            if r.status_code == 200:
                root = ET.fromstring(r.content)
                for s in root.findall('.//akn:speech', ns):
                    # ROBUST EXTRACTION: Try <from>, then <speaker>, then first line
                    from_tag = s.find('akn:from', ns)
                    speaker_tag = s.find('akn:speaker', ns)
                    
                    if from_tag is not None:
                        raw_name = "".join(from_tag.itertext()).strip()
                    elif speaker_tag is not None:
                        raw_name = "".join(speaker_tag.itertext()).strip()
                    else:
                        full_text = "".join(s.itertext()).strip()
                        raw_name = full_text.splitlines()[0] if full_text else "Unknown"

                    speech_pool.append({
                        "date": row["date"],
                        "house_x": row["house"],
                        "speaker": raw_name,
                        "text": "".join(s.itertext()).strip()
                    })
            sleep(0.05)
        except: continue

df_pool = pd.DataFrame(speech_pool)
print(f"Pool created with {len(df_pool)} potential speeches.")

[2010] Fetching speeches...
[2011] Fetching speeches...


In [ ]:
# Normalize the pool
df_pool['speaker_norm'] = df_pool['speaker'].apply(normalize_speaker)

# Map Roles (Taoiseach etc)
def find_role_holder_at(role, when):
    candidates = role_lookup[role_lookup['role'] == role]
    mask = (candidates['start_date'] <= when) & (when <= candidates['end_date'])
    match = candidates.loc[mask]
    return match['fullName_norm'].iat[0] if not match.empty else None

for role_name in role_lookup['role'].unique():
    mask = (df_pool['speaker'] == role_name) & (df_pool['speaker_norm'].isna())
    df_pool.loc[mask, 'speaker_norm'] = df_pool.loc[mask, 'date'].apply(lambda d: find_role_holder_at(role_name, pd.to_datetime(d)))

# Inner Merge (Keeps ONLY people in your members list)
df_members_only = df_pool.merge(df_members, left_on='speaker_norm', right_on='fullName_norm', how='inner')

# Date/Term Validation
df_members_only['date'] = pd.to_datetime(df_members_only['date'])
df_members_only['membership_start'] = pd.to_datetime(df_members_only['membership_start'])
df_members_only['membership_end'] = pd.to_datetime(df_members_only['membership_end'], errors='coerce').fillna(pd.Timestamp.now())

term_mask = (df_members_only['date'] >= df_members_only['membership_start']) & \
            (df_members_only['date'] <= df_members_only['membership_end'])

df_valid_pool = df_members_only[term_mask].copy()
print(f"Total valid member-only speeches found: {len(df_valid_pool)}")

Total valid member-only speeches found: 349965


In [ ]:
TARGET_SIZE = 2224

if len(df_valid_pool) >= TARGET_SIZE:
    # Randomly select the baseline sample
    df_control = df_valid_pool.sample(n=TARGET_SIZE, random_state=42).copy()

    # Replicate Notebook Cleaning Steps
    def remove_speaker_name(text):
        return re.sub(r'^.*?\n', '', text) if isinstance(text, str) else text

    df_control['clean_text'] = df_control['text'].apply(remove_speaker_name)
    df_control['clean_text'] = df_control['clean_text'].str.lower()
    df_control['clean_text'] = df_control['clean_text'].str.replace(r'\n', ' ', regex=True)
    df_control['clean_text'] = df_control['clean_text'].str.replace(r'[^\w\s]', '', regex=True)
    df_control['text_length'] = df_control['clean_text'].apply(lambda x: len(x.split()))

    # Keep relevant columns
    cols_to_keep = ['date', 'house_x', 'speaker', 'text', 'clean_text', 'text_length', 
                    'speaker_norm', 'fullName', 'firstName', 'lastName', 
                    'constituency', 'party', 'membership_start', 'membership_end', 'gender_guess']

    df_control[cols_to_keep].to_csv("data/parliamentary_baseline_cleaned.csv", index=False)
    print(f"Successfully saved {TARGET_SIZE} cleaned member speeches to CSV.")
else:
    print(f"Not enough speeches! Only found {len(df_valid_pool)}. Increase the sample fraction in Chunk 2.")

Successfully saved 2224 cleaned member speeches to CSV.


In [3]:
baseline_df = pd.read_csv("data/parliamentary_baseline_cleaned.csv")

Add a binary column of "in_government" to determine which parties were part of the coalition/the government

In [4]:
# Ensure the date column is in datetime format
baseline_df['date'] = pd.to_datetime(baseline_df['date'])

def is_in_government(row):
    date = row['date']
    party = row['party']
    
    # 2007–2011: Fianna Fáil, Green Party, Progressive Democrats
    if pd.Timestamp('2007-06-14') <= date <= pd.Timestamp('2011-03-08'):
        return 1 if party in ['Fianna Fáil', 'Green Party', 'Progressive Democrats'] else 0
    
    # 2011–2016: Fine Gael, Labour
    elif pd.Timestamp('2011-03-09') <= date <= pd.Timestamp('2016-05-05'):
        return 1 if party in ['Fine Gael', 'Labour Party'] else 0
    
    # 2016–2020: Fine Gael, Independents (Minority)
    elif pd.Timestamp('2016-05-06') <= date <= pd.Timestamp('2020-06-26'):
        return 1 if party in ['Fine Gael', 'Independent'] else 0
    
    # 2020–2024: Fianna Fáil, Fine Gael, Green Party
    elif pd.Timestamp('2020-06-27') <= date <= pd.Timestamp('2025-01-22'):
        return 1 if party in ['Fianna Fáil', 'Fine Gael', 'Green Party'] else 0
    
    # 2025–Present: Fianna Fáil, Fine Gael, Independents
    elif date >= pd.Timestamp('2025-01-23'):
        return 1 if party in ['Fianna Fáil', 'Fine Gael', 'Independent'] else 0
    
    return 0

# Apply the function to create the new column
baseline_df['in_government'] = baseline_df.apply(is_in_government, axis=1)

baseline_df.to_csv("data/parliamentary_baseline_cleaned.csv", index=False) #for R STM 

In [5]:
print(baseline_df['in_government'].value_counts())

in_government
1    1218
0    1006
Name: count, dtype: int64
